In [1]:
import torch.nn.functional as F
import transformers
import torch

model_path = "facebook/layerskip-llama2-7B"

# Initialize tokenizer and model.
tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_path,
    use_safetensors=True,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

def apply_module(module, hidden):
    """Handle Accelerate offloading without moving activations to meta."""
    if module.weight.device.type != "meta":
        hidden = hidden.to(module.weight.device)
    return module(hidden)


text = "The capital of France is"

inputs = tokenizer(
    text,
    return_tensors="pt",
    add_special_tokens=True,
)

embedding_device = model.get_input_embeddings().weight.device
if embedding_device.type != "meta":
    inputs = inputs.to(embedding_device)

num_layers = len(model.model.layers)
results = []

with torch.inference_mode():
    outputs = model.model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        output_hidden_states=True,
        use_cache=False,
        return_dict=True,
    )

    # Final-layer reference distribution.
    # last_hidden_state already includes the final RMSNorm.
    final_hidden = outputs.last_hidden_state[:, -1, :]
    final_logits = apply_module(model.lm_head, final_hidden)

    # Use float32 log_softmax for numerical stability.
    # Keep comparison distributions on CPU to handle device sharding.
    final_log_probs = F.log_softmax(
        final_logits[0].float(), dim=-1
    ).cpu()

    final_token_id = final_log_probs.argmax().item()

    for layer_num in range(1, num_layers + 1):
        if layer_num < num_layers:
            hidden = outputs.hidden_states[layer_num][:, -1, :]
            hidden = apply_module(model.model.norm, hidden)
            logits = apply_module(model.lm_head, hidden)

            layer_log_probs = F.log_softmax(
                logits[0].float(), dim=-1
            ).cpu()
        else:
            layer_log_probs = final_log_probs

        token_id = layer_log_probs.argmax().item()
        probability = layer_log_probs[token_id].exp().item()

        # KL(final || layer), summed over vocabulary for one prompt.
        kl_divergence = F.kl_div(
            input=layer_log_probs,
            target=final_log_probs,
            log_target=True,
            reduction="sum",
        ).item()

        decoded_token = tokenizer.decode(
            [token_id],
            skip_special_tokens=False,
            clean_up_tokenization_spaces=False,
        )

        results.append({
            "layer": layer_num,
            "token_id": token_id,
            "token": decoded_token,
            "probability": probability,
            "kl_final_to_layer": kl_divergence,
            "matches_final": token_id == final_token_id,
        })

print(f"\nPrompt: {text!r}\n")
print(
    f"{'Layer':>5} | {'Token ID':>8} | {'Probability':>11} | "
    f"{'KL(final||layer)':>16} | {'Matches final':>13} | Decoded token"
)
print("-" * 105)

for result in results:
    print(
        f"{result['layer']:>5} | "
        f"{result['token_id']:>8} | "
        f"{result['probability']:>11.4f} | "
        f"{result['kl_final_to_layer']:>16.6f} | "
        f"{str(result['matches_final']):>13} | "
        f"{result['token']!r}"
    )

/home/llminfliu/anaconda3/envs/layer_skip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [01:16<00:00, 25.39s/it]



Prompt: 'The capital of France is'

Layer | Token ID | Probability | KL(final||layer) | Matches final | Decoded token
---------------------------------------------------------------------------------------------------------
    1 |      263 |      0.1298 |         1.894475 |          True | 'a'
    2 |      278 |      0.2019 |         1.702106 |         False | 'the'
    3 |      278 |      0.1776 |         1.533488 |         False | 'the'
    4 |      278 |      0.1167 |         1.485675 |         False | 'the'
    5 |      263 |      0.0663 |         1.473396 |          True | 'a'
    6 |     5982 |      0.0803 |         1.264857 |         False | 'located'
    7 |      278 |      0.0807 |         1.560107 |         False | 'the'
    8 |      263 |      0.0548 |         1.690486 |          True | 'a'
    9 |     5982 |      0.0729 |         1.564948 |         False | 'located'
   10 |      278 |      0.0857 |         1.212524 |         False | 'the'
   11 |      278 |      0.0755 | 

In [4]:
paragraph = "A paragraph is a distinct and separate piece of writing that forms part of an overall text."

In [ ]:
T = 5
words = paragraph.split()
dataset = []

for i in range(0, len(words) - T, T):
    chunk = words[i:i + T]
    target = words[i + 1:i + T + 1]

    dataset.append({
        "chunk": " ".join(chunk),
        "target": " ".join(target),
        "last_layer_token": None,
        "layer_output": None,
        "last_token_match": None
    })



In [12]:
dataset

[{'chunk': 'A paragraph is a distinct',
  'target': 'paragraph is a distinct and'},
 {'chunk': 'and separate piece of writing',
  'target': 'separate piece of writing that'},
 {'chunk': 'that forms part of an', 'target': 'forms part of an overall'}]

In [ ]:
sample_input = dataset[0]["chunk"]
sample_target = dataset[0]["target"]

sample_input_ids = tokenizer(sample_input, return_tensors="pt", add_special_tokens=True)
sample_target_ids = tokenizer(sample_target, return_tensors="pt", add_special_tokens=True)

In [ ]:
for input in sample_input_ids:
    

{'input_ids': tensor([[    1,   319, 14880,   338,   263,  8359]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}

In [ ]:
import pandas as pd

filepath = "cnn_layer_example/v3/test/cnn_dm_test_001.parquet"

df = pd.read_parquet(filepath)
df.head()

In [ ]:
print(df["top10_token_ids"][600])
print(df["top10_probabilities"][600], df["top10_probabilities"][600].sum())

In [ ]:
df[["sequence_id","full_model_prediction","layer"]]